<a href="https://colab.research.google.com/github/SahasransuAcharjya/Text-to-Image-with-XLP-Pipeline/blob/main/text2image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Compatible libraries

In [1]:
!pip install "numpy<2.0"

In [2]:
# Install core AI libraries
!pip install -U diffusers transformers accelerate peft safetensors

# Install face extraction and processing tools
!pip install -q onnxruntime-gpu insightface opencv-python

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.


Backend Engine for the Model

In [1]:
import torch
import gc
import traceback
from PIL import Image
from diffusers import AutoPipelineForText2Image, AutoPipelineForInpainting
from diffusers.utils import load_image
import gradio as gr

# ==========================================
# 1. The Fooocus Style & Resolution Libraries
# ==========================================
FOOOCUS_STYLES = {
    "None": {"prompt": "{prompt}", "negative_prompt": ""},
    "Cinematic": {
        "prompt": "cinematic still {prompt}. emotional, harmonious, vignette, highly detailed, high budget, bokeh, cinemascope, moody, epic, gorgeous, film grain, grainy",
        "negative_prompt": "anime, cartoon, graphic, text, painting, crayon, graphite, abstract, glitch, deformed, mutated, ugly, disfigured"
    },
    "Digital Art": {
        "prompt": "concept art {prompt}. digital artwork, illustrative, painterly, matte painting, highly detailed",
        "negative_prompt": "photo, photorealistic, realism, ugly"
    },
    "Photographic": {
        "prompt": "cinematic photo {prompt}. 35mm photograph, film, professional, 4k, highly detailed",
        "negative_prompt": "drawing, painting, crayon, sketch, graphite, impressionist, noisy, blurry, soft, deformed, ugly"
    },
    "Anime": {
        "prompt": "anime artwork {prompt}. anime style, key visual, vibrant, studio anime, highly detailed",
        "negative_prompt": "photo, deformed, black and white, realism, disfigured, low contrast"
    }
}

FOOOCUS_RESOLUTIONS = {
    "Square (1:1)": (1024, 1024),
    "Portrait (3:4)": (896, 1152),
    "Portrait (9:16)": (768, 1344),
    "Landscape (4:3)": (1152, 896),
    "Landscape (16:9)": (1344, 768)
}

# ==========================================
# 2. The Core Engine (Crash-Proof Version)
# ==========================================
print("Building the AI Engine...")
class CustomImageEngine:
    def __init__(self):
        # Load straight to GPU to avoid System RAM exhaustion
        self.pipe_t2i = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True
        ).to("cuda")

        self.pipe_t2i.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")

        # Share weights to save VRAM
        self.pipe_inpaint = AutoPipelineForInpainting.from_pipe(self.pipe_t2i).to("cuda")

        # THE MAGIC BULLETS: Prevents the massive memory spike during final image decoding
        self.pipe_t2i.vae.enable_tiling()
        self.pipe_t2i.vae.enable_slicing()
        self.pipe_inpaint.vae.enable_tiling()
        self.pipe_inpaint.vae.enable_slicing()

        print("Engine is built and ready in VRAM!")

    def _apply_style(self, prompt, negative_prompt, style_name):
        style_data = FOOOCUS_STYLES.get(style_name, FOOOCUS_STYLES["None"])
        final_prompt = style_data["prompt"].replace("{prompt}", prompt)
        final_negative = style_data["negative_prompt"]
        if negative_prompt:
            final_negative = f"{final_negative}, {negative_prompt}".strip(", ")
        return final_prompt, final_negative

    def _get_dimensions(self, aspect_ratio):
        return FOOOCUS_RESOLUTIONS.get(aspect_ratio, (1024, 1024))

    def generate(self, prompt, style, aspect_ratio, negative_prompt=""):
        styled_prompt, styled_neg = self._apply_style(prompt, negative_prompt, style)
        width, height = self._get_dimensions(aspect_ratio)

        self.pipe_t2i.set_ip_adapter_scale(0.0)
        dummy = Image.new("RGB", (224, 224))

        img = self.pipe_t2i(
            prompt=styled_prompt, negative_prompt=styled_neg, ip_adapter_image=dummy,
            width=width, height=height, num_inference_steps=30
        ).images[0]

        torch.cuda.empty_cache()
        gc.collect()
        return img

    def generate_with_face(self, prompt, face_img_path, style, aspect_ratio, negative_prompt=""):
        styled_prompt, styled_neg = self._apply_style(prompt, negative_prompt, style)
        width, height = self._get_dimensions(aspect_ratio)

        self.pipe_t2i.set_ip_adapter_scale(0.7)
        face_image = load_image(face_img_path).convert("RGB")

        img = self.pipe_t2i(
            prompt=styled_prompt, ip_adapter_image=face_image, negative_prompt=styled_neg,
            width=width, height=height, num_inference_steps=30
        ).images[0]

        torch.cuda.empty_cache()
        gc.collect()
        return img

# Initialize models
engine = CustomImageEngine()

# ==========================================
# 3. The Gradio Web UI
# ==========================================
print("Launching Web Interface...")

def ui_generate(prompt, style, ratio, negative):
    if not prompt: raise gr.Error("Please enter a prompt!")
    try:
        return engine.generate(prompt, style, ratio, negative)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Generation failed: {str(e)}")

def ui_face_swap(prompt, face_img, style, ratio, negative):
    if not prompt: raise gr.Error("Please enter a prompt!")
    if not face_img: raise gr.Error("Please upload a face image!")
    try:
        return engine.generate_with_face(prompt, face_img, style, ratio, negative)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Face swap failed: {str(e)}")

style_choices = list(FOOOCUS_STYLES.keys())
ratio_choices = list(FOOOCUS_RESOLUTIONS.keys())

with gr.Blocks(theme=gr.themes.Soft()) as web_app:
    gr.Markdown("# 🎨 AI Image Studio")
    gr.Markdown("Your custom Fooocus-inspired generation engine.")

    with gr.Tabs():
        # --- TAB 1: Standard Generation ---
        with gr.TabItem("Text to Image"):
            with gr.Row():
                with gr.Column(scale=1):
                    prompt_in = gr.Textbox(label="Prompt", placeholder="A majestic avatar of Lord Vishnu standing in the cosmos...", lines=3)
                    with gr.Row():
                        style_in = gr.Dropdown(choices=style_choices, value="Digital Art", label="Image Style")
                        ratio_in = gr.Dropdown(choices=ratio_choices, value="Square (1:1)", label="Aspect Ratio")
                    neg_in = gr.Textbox(label="Negative Prompt", value="blurry, ugly, distorted")
                    gen_btn = gr.Button("Generate Image", variant="primary")
                with gr.Column(scale=1):
                    img_out = gr.Image(label="Output")

            gen_btn.click(fn=ui_generate, inputs=[prompt_in, style_in, ratio_in, neg_in], outputs=img_out)

        # --- TAB 2: Face Swapping ---
        with gr.TabItem("Face Swapping"):
            with gr.Row():
                with gr.Column(scale=1):
                    face_prompt_in = gr.Textbox(label="Prompt", placeholder="A portrait of an astronaut...", lines=3)
                    face_upload = gr.Image(label="Upload Face Reference", type="filepath")
                    with gr.Row():
                        face_style_in = gr.Dropdown(choices=style_choices, value="Photographic", label="Image Style")
                        face_ratio_in = gr.Dropdown(choices=ratio_choices, value="Portrait (3:4)", label="Aspect Ratio")
                    face_neg_in = gr.Textbox(label="Negative Prompt", value="blurry, distorted")
                    face_btn = gr.Button("Generate with Face", variant="primary")
                with gr.Column(scale=1):
                    face_out = gr.Image(label="Output")

            face_btn.click(fn=ui_face_swap, inputs=[face_prompt_in, face_upload, face_style_in, face_ratio_in, face_neg_in], outputs=face_out)

web_app.launch(share=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Building the AI Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: h94/IP-Adapter
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Engine is built and ready in VRAM!
Launching Web Interface...


/tmp/ipykernel_33459/3766555743.py:138: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as web_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://550eaf0ef114a53537.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
